# Data Exploration — TriageIQ

Initial EDA on scraped GitHub Issues.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')


In [ ]:
df = pd.read_parquet('../data/processed/issues_microsoft_vscode.parquet')
print(f'Total issues: {len(df)}')
print(f'Date range: {df.created_at.min()} — {df.created_at.max()}')
print(f'State: {df.state.value_counts().to_dict()}')


In [ ]:
# Label distribution
from collections import Counter
all_labels = [l for lbls in df.labels_raw for l in lbls]
top_labels = Counter(all_labels).most_common(30)
print('Top 30 labels:')
for label, count in top_labels:
    print(f'  {label:40s} {count:>6}')


In [ ]:
# Resolution time distribution
closed = df[df.resolution_hours.notna()]
print(f'Closed issues: {len(closed)}')
print(closed.resolution_hours.describe())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(closed.resolution_hours.clip(upper=30*24), bins=50)
ax.set_xlabel('Resolution time (hours, clipped at 30 days)')
ax.set_title('Resolution Time Distribution — microsoft/vscode')
plt.tight_layout()
plt.savefig('../reports/fig_resolution_time_vscode.png', dpi=150)
plt.show()


In [ ]:
# Text length histogram
df['body_len'] = df.body_clean.str.len()
print('Body length stats:')
print(df.body_len.describe())


In [ ]:
# Save summary stats
summary = {
    'total_issues': len(df),
    'date_start': str(df.created_at.min()),
    'date_end': str(df.created_at.max()),
    'closed_count': int(df.state.eq('closed').sum()),
    'open_count': int(df.state.eq('open').sum()),
    'median_resolution_hours': float(closed.resolution_hours.median()),
    'top_labels': top_labels[:10],
}

import json
with open('../reports/01_eda_initial.md', 'w') as f:
    f.write('# Initial EDA — microsoft/vscode\n\n')
    for k, v in summary.items():
        f.write(f'- **{k}**: {v}\n')

print('Saved reports/01_eda_initial.md')
